# Step 7: Streaming Inference & Drift Simulation

Simulate streaming patient data, run real-time inference against the deployed SPCS endpoint, and monitor for drift detection alerts.

## Capabilities

| Feature | Description |
|---------|-------------|
| **Streaming Simulation** | Generates patient data at configurable intervals |
| **Real-Time Inference** | Calls deployed REST endpoint via Model Registry |
| **Drift Injection** | Introduces data drift after configurable delay |
| **Alert Monitoring** | Continuously checks for triggered drift alerts |

## Prerequisites

- Run notebooks 01-06 first
- SPCS inference service must be running (notebook 05)
- Model monitor and alerts must be configured (notebook 06)

## Imports and Configuration

In [1]:
%cd ..
%load_ext autoreload

/Users/ccaudill/src/github/snowflake-ml-prod


In [2]:
%autoreload
import os
import sys
import time
import logging
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from IPython.display import display, clear_output

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

from snowflake.snowpark import Session
from snowflake.ml.registry import Registry
from source.configs import get_config
from source.utils import get_session, get_feature_config
from source.framework.deploy import ModelDeployer
from data.simulator import StreamingDataSimulator, DRIFT_TYPES

config = get_config("source/config.yaml")
session = get_session(config.snowflake.connection_name)

DB = config.snowflake.database
SCHEMA = config.snowflake.schema_name
COMPUTE_WAREHOUSE = config.snowflake.warehouse

session.use_database(DB)
session.use_schema(SCHEMA)
session.use_warehouse(COMPUTE_WAREHOUSE)

print(f"Connected as: {session.get_current_user()}")
print(f"Current role: {session.get_current_role()}")
print(f"Current warehouse: {session.get_current_warehouse()}")

2026-04-30 17:38:30,504 - INFO - AST state has not been set explicitly. Defaulting to ast_enabled = True.
/Users/ccaudill/.pyenv/versions/snowflake-ml-prod/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-30 17:38:32,658 - INFO - Snowflake Connector for Python Version: 4.4.0, Python Version: 3.11.14, Platform: macOS-26.4-arm64-arm-64bit
2026-04-30 17:38:32,658 - INFO - Connecting to GLOBAL Snowflake domain


Creating Session...


2026-04-30 17:38:35,685 - INFO - Snowpark Session information: 
"version" : 1.50.0,
"python.version" : 3.11.14,
"python.connector.version" : 4.4.0,
"python.connector.session.id" : 5727214346269250,
"os.name" : Darwin



Connected as: "CCAUDILL"
Current role: "SYSADMIN"
Current warehouse: "ML_DEMO_WAREHOUSE"


## Configurable Parameters

Adjust these values to control simulation behavior.

In [3]:
SIMULATION_DURATION_SECONDS = 600
INTERVAL_BETWEEN_BATCHES_SECONDS = 2
RECORDS_PER_BATCH = 10
DRIFT_AFTER_SECONDS = 240
DRIFT_TYPE = "vital_degradation"
ALERT_CHECK_INTERVAL_BATCHES = 5

print("=== Simulation Configuration ===")
print(f"  Duration:              {SIMULATION_DURATION_SECONDS}s ({SIMULATION_DURATION_SECONDS / 60:.1f} min)")
print(f"  Batch interval:        {INTERVAL_BETWEEN_BATCHES_SECONDS}s")
print(f"  Records per batch:     {RECORDS_PER_BATCH}")
print(f"  Drift after:           {DRIFT_AFTER_SECONDS}s")
print(f"  Drift type:            {DRIFT_TYPE}")
print(f"  Alert check every:     {ALERT_CHECK_INTERVAL_BATCHES} batches")
print(f"  Est. total records:    ~{int(SIMULATION_DURATION_SECONDS / INTERVAL_BETWEEN_BATCHES_SECONDS) * RECORDS_PER_BATCH}")
print(f"\nAvailable drift types: {list(DRIFT_TYPES.keys())}")

=== Simulation Configuration ===
  Duration:              600s (10.0 min)
  Batch interval:        2s
  Records per batch:     10
  Drift after:           240s
  Drift type:            vital_degradation
  Alert check every:     5 batches
  Est. total records:    ~3000

Available drift types: ['age_shift', 'vital_degradation', 'feature_scale', 'distribution_shift']


## Initialize Model & Deployer

In [4]:
MODEL_NAME = config.model.model_name
SERVICE_NAME = config.deploy.service_name

registry = Registry(session, database_name=DB, schema_name=SCHEMA)
model = registry.get_model(MODEL_NAME)
versions = model.versions()
MODEL_VERSION = versions[-1].version_name

deployer = ModelDeployer(session=session, registry_database=DB, registry_schema=SCHEMA)

service_status = deployer.get_service_status(SERVICE_NAME)
print(f"Model: {MODEL_NAME}")
print(f"Version: {MODEL_VERSION}")
print(f"Service: {SERVICE_NAME} ({service_status})")

if service_status != "RUNNING":
    print("\nWARNING: Service is not RUNNING. Deploy the model first (notebook 05).")

Model: PATIENT_RISK_MODEL
Version: V_20260430_221213
Service: PATIENT_RISK_SERVICE (RUNNING)


## Feature Engineering Helper

Compute the same engineered features the model expects from raw simulator output.

In [5]:
feature_config = get_feature_config(config)
FEATURE_COLUMNS = [c.upper() for c in feature_config["all_numeric_features"] + feature_config["all_categorical_features"]]

# compute_engineered_features = StreamingDataSimulator.compute_engineered_features

print(f"Feature columns for inference ({len(FEATURE_COLUMNS)}): {FEATURE_COLUMNS}")

Feature columns for inference (23): ['AGE', 'BMI', 'HEART_RATE', 'SYSTOLIC_BP', 'DIASTOLIC_BP', 'TEMPERATURE', 'RESPIRATORY_RATE', 'OXYGEN_SATURATION', 'GLUCOSE_LEVEL', 'CREATININE', 'HEMOGLOBIN', 'WBC_COUNT', 'COMORBIDITY_COUNT', 'PREVIOUS_ADMISSIONS', 'MEDICATION_COUNT', 'SHOCK_INDEX', 'PULSE_PRESSURE', 'VITAL_SIGNS_SEVERITY', 'GENDER', 'PRIMARY_DIAGNOSIS', 'ADMISSION_TYPE', 'INSURANCE_TYPE', 'BMI_CATEGORY']


## Drift Alert Checker

Query the alert history to detect if any drift alerts have fired.

In [6]:
MONITOR_NAME = f"{MODEL_NAME}_MONITOR"
DRIFT_ALERT_COLUMNS = ["AGE", "HEART_RATE", "GLUCOSE_LEVEL"]


def check_drift_alerts(since: datetime) -> pd.DataFrame:
    alert_names = [f"{MONITOR_NAME}_{col}_DRIFT_ALERT" for col in DRIFT_ALERT_COLUMNS]
    alert_list = ", ".join(f"'{a}'" for a in alert_names)

    query = f"""
    SELECT
        NAME,
        STATE,
        CONDITION_TEXT,
        LAST_TRIGGERED,
        LAST_TRIGGERED_TIMESTAMP
    FROM TABLE(INFORMATION_SCHEMA.ALERT_HISTORY(
        SCHEDULED_TIME_RANGE_START => '{since.strftime("%Y-%m-%dT%H:%M:%SZ")}'
    ))
    WHERE NAME IN ({alert_list})
    ORDER BY LAST_TRIGGERED_TIMESTAMP DESC
    """
    try:
        rows = session.sql(query).collect()
        if rows:
            return pd.DataFrame([r.as_dict() for r in rows])
    except Exception:
        pass
    return pd.DataFrame()


def check_alert_status() -> pd.DataFrame:
    results = []
    for col in DRIFT_ALERT_COLUMNS:
        alert_name = f"{MONITOR_NAME}_{col}_DRIFT_ALERT"
        try:
            rows = session.sql(f"DESCRIBE ALERT {alert_name}").collect()
            if rows:
                row = rows[0].as_dict()
                results.append({
                    "Alert": alert_name,
                    "State": row.get("state", "UNKNOWN"),
                    "Schedule": row.get("schedule", "N/A"),
                })
        except Exception as e:
            results.append({"Alert": alert_name, "State": f"ERROR: {e}", "Schedule": "N/A"})
    return pd.DataFrame(results)


print("Current alert status:")
display(check_alert_status())

Current alert status:


,Alert,State,Schedule
0,PATIENT_RISK_MODEL_MONITOR_AGE_DRIFT_ALERT,started,1 MINUTE
1,PATIENT_RISK_MODEL_MONITOR_HEART_RATE_DRIFT_ALERT,started,1 MINUTE
2,PATIENT_RISK_MODEL_MONITOR_GLUCOSE_LEVEL_DRIFT...,started,1 MINUTE


## Run Streaming Inference Simulation

This loop:
1. Generates a batch of patient records using the simulator
2. Computes engineered features
3. Runs inference against the deployed SPCS endpoint
4. Inserts records (with predictions) into `STREAMING_PATIENT_DATA`
5. Introduces drift after the configured delay
6. Periodically checks for triggered drift alerts

In [15]:
start_time = time.time()
sim_start = datetime.now()
batch_count = 0
total_records = 0
drift_active = False
alerts_triggered = []
auth_token = os.environ["SNOWFLAKE_TOKEN"]

print(f"Starting streaming inference simulation at {sim_start.strftime('%H:%M:%S')}")
print(f"  Duration: {SIMULATION_DURATION_SECONDS}s | Batch interval: {INTERVAL_BETWEEN_BATCHES_SECONDS}s")
print(f"  Drift ({DRIFT_TYPE}) will activate after {DRIFT_AFTER_SECONDS}s")
print("=" * 80)

simulator = StreamingDataSimulator(
    session=session,
    database=DB,
    schema_name=SCHEMA,
)

try:
    while (time.time() - start_time) < SIMULATION_DURATION_SECONDS:
        elapsed = time.time() - start_time
        batch_count += 1

        if not drift_active and elapsed >= DRIFT_AFTER_SECONDS:
            drift_active = True
            simulator.enable_drift(DRIFT_TYPE)
            print(f"\n{'!' * 60}")
            print(f"  DRIFT ENABLED at {elapsed:.0f}s — type: {DRIFT_TYPE}")
            print(f"{'!' * 60}\n")

        # == Generate Simulation Data Batch ==
        batch_df = simulator.generate_batch(
            batch_size=RECORDS_PER_BATCH,
            compute_features=True,
        )
        try:
            predictions = deployer.predict_rest(
                service_name=SERVICE_NAME,
                features_df=batch_df,
                endpoint_path="/predict",
                token=auth_token
            )
            pred_values = [d[1]["output_feature_0"] for d in predictions['data']]
            batch_df["PREDICTED_RISK_LEVEL"] = pred_values
        except Exception as e:
            logger.warning(f"Inference failed for batch {batch_count}: {e}")
            batch_df["PREDICTED_RISK_LEVEL"] = "ERROR"

        total_records += len(batch_df)

        pred_dist = batch_df["PREDICTED_RISK_LEVEL"].value_counts().to_dict()
        drift_marker = " [DRIFT]" if drift_active else ""
        print(
            f"Batch {batch_count:>4d} | "
            f"{elapsed:>6.0f}s | "
            f"Records: {total_records:>6d} | "
            f"Predictions: {pred_dist}{drift_marker}"
        )

        # == Check for Drift Alerts ==
        if batch_count % ALERT_CHECK_INTERVAL_BATCHES == 0:
            alert_df = check_drift_alerts(sim_start)
            if not alert_df.empty:
                new_alerts = alert_df[~alert_df["NAME"].isin(alerts_triggered)]
                if not new_alerts.empty:
                    print(f"\n{'*' * 60}")
                    print(f"  DRIFT ALERTS TRIGGERED!")
                    for _, row in new_alerts.iterrows():
                        print(f"    - {row['NAME']} at {row.get('LAST_TRIGGERED_TIMESTAMP', 'N/A')}")
                        alerts_triggered.append(row["NAME"])
                    print(f"{'*' * 60}\n")
            else:
                print(f"  [Alert check @ batch {batch_count}] No drift alerts triggered yet")

        time.sleep(INTERVAL_BETWEEN_BATCHES_SECONDS)

except KeyboardInterrupt:
    print("\nSimulation interrupted by user")

end_time = time.time()
duration = end_time - start_time

print("\n" + "=" * 80)
print(f"Simulation complete")
print(f"  Duration:        {duration:.1f}s")
print(f"  Total batches:   {batch_count}")
print(f"  Total records:   {total_records}")
print(f"  Drift enabled:   {drift_active} ({DRIFT_TYPE})")
print(f"  Alerts fired:    {len(alerts_triggered)}")

2026-04-30 17:46:00,825 - INFO - REST inference: url=https://nua43ycf-sfsenorthamerica-ccaudill-aws2.snowflakecomputing.app/predict, rows=10


Starting streaming inference simulation at 17:46:00
  Duration: 600s | Batch interval: 2s
  Drift (vital_degradation) will activate after 240s


2026-04-30 17:46:01,474 - INFO - REST inference returned: {'data': [[0, {'output_feature_0': 'MEDIUM'}], [1, {'output_feature_0': 'HIGH'}], [2, {'output_feature_0': 'LOW'}], [3, {'output_feature_0': 'MEDIUM'}], [4, {'output_feature_0': 'MEDIUM'}], [5, {'output_feature_0': 'MEDIUM'}], [6, {'output_feature_0': 'MEDIUM'}], [7, {'output_feature_0': 'CRITICAL'}], [8, {'output_feature_0': 'LOW'}], [9, {'output_feature_0': 'CRITICAL'}]]}


Batch    1 |      0s | Records:     10 | Predictions: {'MEDIUM': 5, 'LOW': 2, 'CRITICAL': 2, 'HIGH': 1}


2026-04-30 17:46:03,488 - INFO - REST inference: url=https://nua43ycf-sfsenorthamerica-ccaudill-aws2.snowflakecomputing.app/predict, rows=10
2026-04-30 17:46:04,047 - INFO - REST inference returned: {'data': [[0, {'output_feature_0': 'MEDIUM'}], [1, {'output_feature_0': 'CRITICAL'}], [2, {'output_feature_0': 'CRITICAL'}], [3, {'output_feature_0': 'MEDIUM'}], [4, {'output_feature_0': 'MEDIUM'}], [5, {'output_feature_0': 'MEDIUM'}], [6, {'output_feature_0': 'HIGH'}], [7, {'output_feature_0': 'MEDIUM'}], [8, {'output_feature_0': 'MEDIUM'}], [9, {'output_feature_0': 'HIGH'}]]}


...

Batch   54 |    131s | Records:    540 | Predictions: {'MEDIUM': 6, 'LOW': 4}


2026-04-30 17:48:14,015 - INFO - REST inference: url=https://nua43ycf-sfsenorthamerica-ccaudill-aws2.snowflakecomputing.app/predict, rows=10
2026-04-30 17:48:14,411 - INFO - REST inference returned: {'data': [[0, {'output_feature_0': 'MEDIUM'}], [1, {'output_feature_0': 'LOW'}], [2, {'output_feature_0': 'MEDIUM'}], [3, {'output_feature_0': 'MEDIUM'}], [4, {'output_feature_0': 'MEDIUM'}], [5, {'output_feature_0': 'MEDIUM'}], [6, {'output_feature_0': 'LOW'}], [7, {'output_feature_0': 'MEDIUM'}], [8, {'output_feature_0': 'LOW'}], [9, {'output_feature_0': 'CRITICAL'}]]}


Batch   55 |    133s | Records:    550 | Predictions: {'MEDIUM': 6, 'LOW': 3, 'CRITICAL': 1}
  [Alert check @ batch 55] No drift alerts triggered yet


2026-04-30 17:48:17,018 - INFO - REST inference: url=https://nua43ycf-sfsenorthamerica-ccaudill-aws2.snowflakecomputing.app/predict, rows=10
2026-04-30 17:48:17,308 - INFO - REST inference returned: {'data': [[0, {'output_feature_0': 'LOW'}], [1, {'output_feature_0': 'MEDIUM'}], [2, {'output_feature_0': 'LOW'}], [3, {'output_feature_0': 'LOW'}], [4, {'output_feature_0': 'HIGH'}], [5, {'output_feature_0': 'MEDIUM'}], [6, {'output_feature_0': 'HIGH'}], [7, {'output_feature_0': 'MEDIUM'}], [8, {'output_feature_0': 'MEDIUM'}], [9, {'output_feature_0': 'LOW'}]]}


Batch   56 |    136s | Records:    560 | Predictions: {'LOW': 4, 'MEDIUM': 4, 'HIGH': 2}

Simulation interrupted by user

Simulation complete
  Duration:        136.7s
  Total batches:   56
  Total records:   560
  Drift enabled:   False (vital_degradation)
  Alerts fired:    0


## Post-Simulation Analysis

In [ ]:
print("=== Streaming Data Summary ===")
row_count = session.sql(f"SELECT COUNT(*) AS CNT FROM {DB}.{SCHEMA}.{STREAMING_TABLE}").collect()[0]["CNT"]
print(f"Total rows in {STREAMING_TABLE}: {row_count}")

print("\n--- Prediction Distribution (Overall) ---")
session.sql(f"""
SELECT "PREDICTED_RISK_LEVEL", COUNT(*) AS COUNT,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS PCT
FROM {DB}.{SCHEMA}.{STREAMING_TABLE}
GROUP BY "PREDICTED_RISK_LEVEL"
ORDER BY COUNT DESC
""").show()

print("--- Prediction Distribution (Pre-Drift vs Post-Drift) ---")
session.sql(f"""
SELECT
    "DRIFT_APPLIED",
    "PREDICTED_RISK_LEVEL",
    COUNT(*) AS COUNT
FROM {DB}.{SCHEMA}.{STREAMING_TABLE}
GROUP BY "DRIFT_APPLIED", "PREDICTED_RISK_LEVEL"
ORDER BY "DRIFT_APPLIED", COUNT DESC
""").show()

print("--- Key Feature Means (Pre-Drift vs Post-Drift) ---")
session.sql(f"""
SELECT
    "DRIFT_APPLIED",
    ROUND(AVG("AGE"), 1) AS AVG_AGE,
    ROUND(AVG("HEART_RATE"), 1) AS AVG_HR,
    ROUND(AVG("SYSTOLIC_BP"), 1) AS AVG_SBP,
    ROUND(AVG("OXYGEN_SATURATION"), 1) AS AVG_SPO2,
    ROUND(AVG("GLUCOSE_LEVEL"), 1) AS AVG_GLUCOSE,
    ROUND(AVG("CREATININE"), 2) AS AVG_CREATININE
FROM {DB}.{SCHEMA}.{STREAMING_TABLE}
GROUP BY "DRIFT_APPLIED"
ORDER BY "DRIFT_APPLIED"
""").show()

## Final Drift Alert Check

In [ ]:
print("=== Drift Alert Status ===")
display(check_alert_status())

print("\n=== Alert History (since simulation start) ===")
alert_history = check_drift_alerts(sim_start)
if not alert_history.empty:
    display(alert_history)
else:
    print("No alerts have fired yet.")
    print("Note: Alerts run on a schedule (default 60 min). Drift may not be detected immediately.")
    print("Re-run this cell later or check in Snowsight under Monitoring > Alerts.")

## Check Monitor Drift Metrics

In [ ]:
print("=== Monitor Status ===")
result = session.sql(f"DESC MODEL MONITOR {MONITOR_NAME}").collect()
if result:
    row = result[0]
    print(f"Monitor: {MONITOR_NAME}")
    print(f"  State: {row['monitor_state']}")
    print(f"  Aggregation Status: {row['aggregation_status']}")

print("\n=== Drift Metrics (PSI) for Key Features ===")
for col in ["AGE", "HEART_RATE", "SYSTOLIC_BP", "OXYGEN_SATURATION", "GLUCOSE_LEVEL"]:
    try:
        drift_df = session.sql(f"""
        SELECT *
        FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
            '{MONITOR_NAME}',
            'PSI',
            '{col}',
            'DAY',
            DATEADD('day', -1, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
            CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
        ))
        """).collect()
        if drift_df:
            val = drift_df[0].as_dict().get("VALUE", "N/A")
            flag = " *** DRIFTED ***" if isinstance(val, (int, float)) and val > 0.2 else ""
            print(f"  {col:<25s} PSI = {val}{flag}")
        else:
            print(f"  {col:<25s} No data yet")
    except Exception as e:
        print(f"  {col:<25s} Error: {e}")

## Summary

| Object | Type | Purpose |
|--------|------|---------|
| `STREAMING_PATIENT_DATA` | Table | Simulated streaming records with predictions |
| `StreamingDataSimulator` | Python | Generates realistic patient data with drift injection |
| `ModelDeployer.predict()` | Python | Calls SPCS REST endpoint via Model Registry |
| `*_DRIFT_ALERT` | Alerts | Monitored during simulation for triggered drift |

## Next Step

Continue to **08_cleanup.ipynb** to tear down resources.